In [0]:
# O QUE: Analisar a tabela Bronze de pedidos.
# POR QUÊ: Identificar duplicidades e valores nulos
# antes da transformação para Silver.

from pyspark.sql import functions as F

df_orders = spark.table("workspace.bronze.orders")

total = df_orders.count()
ids_unicos = df_orders.select("order_id").distinct().count()

print(f"Total de registros: {total}")
print(f"IDs únicos: {ids_unicos}")
print(f"IDs repetidos: {total - ids_unicos}")

# Contagem de valores nulos por coluna
df_nulos = df_orders.select([
    F.sum(
        F.when(
            F.col(c).isNull() | (F.trim(F.col(c)) == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in df_orders.columns
])

display(df_nulos)


In [0]:

# O QUE: Converter as datas da Bronze para timestamp.
# POR QUÊ: Preparar os pedidos para análises e validações.

from pyspark.sql import functions as F

colunas_data = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

df_orders_silver = df_orders

for coluna in colunas_data:
    df_orders_silver = df_orders_silver.withColumn(
        coluna,
        F.try_to_timestamp(F.col(coluna))
    )

# Inspecionar os tipos resultantes
df_orders_silver.printSchema()

# Validar a quantidade
print("Registros Silver:", df_orders_silver.count())


In [0]:

# O QUE: Identificar datas que falharam na conversão.
# POR QUÊ: Impedir perda silenciosa de informações.

from pyspark.sql import functions as F

resultados = []

for coluna in colunas_data:
    invalidos = df_orders.join(
        df_orders_silver.select(
            "order_id",
            F.col(coluna).alias("data_convertida")
        ),
        on="order_id"
    ).filter(
        F.col(coluna).isNotNull()
        & (F.trim(F.col(coluna)) != "")
        & F.col("data_convertida").isNull()
    ).count()

    resultados.append((coluna, invalidos))

display(
    spark.createDataFrame(
        resultados,
        ["coluna", "conversoes_invalidas"]
    )
)


In [0]:

# O QUE: Persistir os pedidos tratados na camada Silver.
# POR QUÊ: Disponibilizar dados tipados para análises futuras.

tabela_silver = "workspace.silver.orders"

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

# Impedir sobrescrita de uma tabela já existente
if spark.catalog.tableExists(tabela_silver):
    print(f"Tabela já existe: {tabela_silver}")
    print("Nenhuma gravação realizada.")
else:
    (
        df_orders_silver.write
        .format("delta")
        .saveAsTable(tabela_silver)
    )
    print(f"Tabela criada: {tabela_silver}")

# Validar o resultado
print(f"Registros: {spark.table(tabela_silver).count()}")

display(
    spark.sql(f"DESCRIBE DETAIL {tabela_silver}")
    .select("format", "numFiles", "sizeInBytes")
)
